In [1]:
import pandas as pd
import numpy as np
import yaml
from collections import defaultdict

## Load data

In [2]:
path_to_file = "data/mmc.yaml"

with open(path_to_file, "r", encoding="utf-8") as file:
    raw_data = yaml.safe_load(file)
    
df = pd.DataFrame(raw_data)

df_naselja = pd.read_csv("./processed_data/naselja.csv")

df_regije = df_naselja[['region_id', 'region_name']].drop_duplicates().values



In [3]:
#df = df.sample(frac=0.1) #Keep only 10% of all data for testing

In [4]:
print(f"Vseh člankov: {len(df)}")
df = df[df['paragraphs'].str.len() > 0]
empty_values = ["", "[]"]
df = df[~df['topics'].isin(empty_values)]

Vseh člankov: 73461


In [5]:
# Clean text column, kjer je vse lower case in nima stopwordov

with open('./data/stopwords-sl.txt', 'r', encoding='utf-8') as f:
    sl_stopwords = [line.strip() for line in f if line.strip()]

def clean_data(text_lst):
    full_text = " ".join(text_lst)
    clean_text = "".join(c.lower() if c.isalpha() or c in "čšžćđ-" else " " for c in full_text)
    padded_text = f" {clean_text} "    
    words = padded_text.split()
    final_words = [w for w in words if w not in sl_stopwords]
    return " ".join(final_words)

df["clean_text"] = df["paragraphs"].apply(clean_data)
df["paragraphs"] = df["paragraphs"].apply(
    lambda ps: "\n\n".join(ps) if isinstance(ps, list) else str(ps)
)

In [6]:
# Odstrani naselja, ki imajo več kot 1 lokacijo 

region_counts = df_naselja.groupby('naselje')['region_name'].transform('nunique')
ambiguous_mask = region_counts > 1
df_removed = df_naselja[ambiguous_mask].sort_values(by='naselje')
df_naselja = df_naselja[~ambiguous_mask].copy()
df_naselja = df_naselja.drop_duplicates(subset=['naselje', 'region_name'])

In [7]:
df.head(3)

,_id,url,authors,date,title,paragraphs,figures,lead,mention,topics,keywords,gpt_keywords,id,n_comments,category,clean_text
0,64613b29e32ccd3c085ab743,https://www.rtvslo.si/sport/kosarka/liga-nba/v...,[T. J.],2023-05-14T18:17:00,V javnost prišel nov posnetek Moranta s pištolo,"Moštvo iz Tennesseeja je ob tem zapisalo, da j...",[{'caption': 'Ja Morant je eden najatraktivnej...,Po družbenih omrežjih je zakrožil nov posnetek...,[],sport,"[Ja Morant, Memphis Grizzlies, Liga NBA]","[Jaja Morant, košarka, pištola, Memphis Grizzl...",668124,77.0,NaN,moštvo tennesseeja tem zapisalo morant suspend...
1,64613b29e32ccd3c085ab744,https://www.rtvslo.si/slovenija/klakocar-zupan...,[La. Da.],2023-05-12T13:41:00,Klakočar Zupančič: DZ je v zadnjem letu delova...,"Aktualna sestava 90-članskega državnega zbora,...",[{'img': 'https://img.rtvcdn.si/_up/upload/202...,"V soboto bo minilo eno leto, odkar se je na us...","[Roberta Goloba, Tanjo Fajon, Tatjano Bobnar]",slovenija,"[DZ, deveti sklic, poslanci, seje, obletnica]","[državni zbor, ocena, efektivnost, seja, zakon...",667927,39.0,NaN,aktualna sestava -članskega državnega zbora dv...
2,64613b29e32ccd3c085ab745,https://www.rtvslo.si/kultura/beremo/andrej-e-...,"[Katja Šifkovič, Program Ars]",2023-05-06T10:21:00,Andrej E. Skubic: Pa čeprav buldožer,"Citat "" Ako želiš biću ja, program tvog kompju...",[{'img': 'https://img.rtvcdn.si/_up/upload/202...,Hrvaški duo Denis&Denis vnovič osvaja slovensk...,[],kultura,"[Andrej E. Skubic, Založba Goga, roman]","[Denis&Denis, roman, Andrej Skubic, kresnik, p...",667216,1.0,NaN,citat ako želiš biću ja program tvog kompjuter...


## Find regions for each data

In [8]:
#! FOUND EVEN FASTER WAY

# import re
# from collections import defaultdict

# # ─────────────────────────────────────────────────────────────────────────────
# # 1. Pre‑build mapping: term → set of region_id (once, before the loop)
# # ─────────────────────────────────────────────────────────────────────────────
# term_to_region = defaultdict(set)

# for _, row in df_naselja.iterrows():
#     rid = row['region_id']
#     for col in ('naselje', 'rodilnik', 'mestnik'):
#         term = str(row[col])
#         if term and term.lower() != 'nan':
#             term_to_region[term].add(rid)   # keep original case

# # ─────────────────────────────────────────────────────────────────────────────
# # 2. Build regex pattern from all terms (same as before)
# # ─────────────────────────────────────────────────────────────────────────────
# all_terms = set(term_to_region.keys())
# pattern = '|'.join(re.escape(term) for term in all_terms)
# regex = re.compile(rf'\b({pattern})\b', flags=re.IGNORECASE)

# # ─────────────────────────────────────────────────────────────────────────────
# # 3. Fast lookup function – no DataFrame scan per row
# # ─────────────────────────────────────────────────────────────────────────────
# def get_intersected_regions_fast(text_list):
#     full_text = " ".join(text_list)
#     found_words = set(regex.findall(full_text))

#     if not found_words:
#         return None

#     # Collect region IDs directly from the pre‑built mapping
#     region_ids = set()
#     for word in found_words:
#         # Keep original case for lookup (map is case‑sensitive)
#         # But regex.findall with IGNORECASE returns the word *as found*.
#         # If your terms have different case variants, unify them:
#         # word_lower = word.lower()
#         # for key in term_to_region: if key.lower() == word_lower ...
#         # Simpler: store mapping with lower‑case keys and lower‑case the found word.
#         #
#         # The clearest approach (avoids case mismatch):
#         # Build term_to_region with lower‑case keys, then use word.lower().
#         region_ids.update(term_to_region.get(word, set()))
#     return list(region_ids)

# # 4. Apply
# df['intersected_regions'] = df['paragraphs'].apply(get_intersected_regions_fast)

### Brez naselji

In [9]:
# term_to_region = defaultdict(set)

# for _, row in df_naselja.iterrows():
#     rid = row['region_id']
#     for col in ('naselje', 'rodilnik', 'mestnik'):
#         term = str(row[col])
#         if term and term.lower() != 'nan':
#             term_to_region[term].add(rid)

In [10]:
# import ahocorasick

# # Build automaton with lowercase terms
# automaton = ahocorasick.Automaton()
# for term, region_ids in term_to_region.items():
#     spaced_term1 = f" {term.lower()} " # Beseda more imeti space okrog sebe
#     automaton.add_word(spaced_term1, (list(region_ids), term))  # Store just the IDs
# automaton.make_automaton()

# def get_regions_ahocorasick_fixed(text):
#     region_ids = set()
#     found_matches = {} 
    
#     for _, (ids, word) in automaton.iter(text):
#         region_ids.update(ids)
#         found_matches[word] = ids

#     #if found_matches:
#         # Create a list of "Word (ID, ID)" strings
#     #    display_list = [f"{word} {ids}" for word, ids in found_matches.items()]
#     #    print(f"Matched: {', '.join(display_list)}")
    
#     return list(region_ids) if region_ids else None
# df['intersected_regions'] = df['clean_text'].apply(get_regions_ahocorasick_fixed)


### Get also the naselja

In [11]:
from collections import defaultdict

term_to_data = {}

for _, row in df_naselja.iterrows():
    rid = row['region_id']
    base_naselje = str(row['naselje'])
    
    for col in ('naselje', 'rodilnik', 'mestnik'):
        term = str(row[col])
        if term and term.lower() != 'nan':
            term_lower = term.lower()
            
            # Store the tuple directly as the target payload
            if term_lower not in term_to_data:
                term_to_data[term_lower] = (rid, base_naselje)

In [12]:
import ahocorasick

automaton = ahocorasick.Automaton()

for term_lower, location_tuple in term_to_data.items():
    spaced_term = f" {term_lower} " 
    automaton.add_word(spaced_term, location_tuple)

automaton.make_automaton()

In [13]:
def get_regions_and_naselja_tuples(text):
    region_ids = set()
    matched_tuples = set()

    text = f" {str(text).lower()} "

    for _, (rid, base_naselje) in automaton.iter(text):
        region_ids.add(rid)
        matched_tuples.add((rid, base_naselje))

    if not region_ids:
        return None, None

    return list(region_ids), list(matched_tuples)

# 4. Apply to your dataframe
res = df['clean_text'].apply(get_regions_and_naselja_tuples)

df['intersected_regions'] = res.apply(lambda x: x[0])
df['intersected_naselja'] = res.apply(lambda x: x[1])

novice_z_naselji_df = df[df['intersected_regions'].notna()].copy()

print(f"Člankov z zaznano regijo: {len(novice_z_naselji_df)}")
print(f"Pokritost: {len(novice_z_naselji_df) / len(df):.2%}")

Člankov z zaznano regijo: 28579
Pokritost: 39.45%


In [14]:
novice_z_naselji_df

,_id,url,authors,date,title,paragraphs,figures,lead,mention,topics,keywords,gpt_keywords,id,n_comments,category,clean_text,intersected_regions,intersected_naselja
4,64613b29e32ccd3c085ab747,https://www.rtvslo.si/kultura/drugo/slovenec-k...,[P. G.],2023-05-10T08:51:00,"""Slovenec, ki je preletel stoletje"": odkrili k...","Jurij Kraigher (1891, Hrašče pri Postojni) je ...",[{'img': 'https://img.rtvcdn.si/_up/upload/202...,V Hraščah po novem stoji doprsni kip pilota in...,"[Stanko Bloudek, Katja Majer, Srečko Šajn]",kultura,"[Jurij Kraigher, kip, letalstvo]","[Jurij Kraigher, pilot, inovator, letalstvo, k...",667600,7.0,NaN,jurij kraigher hrašče postojni vojaški pilot s...,[SI038],"[(SI038, Pivka), (SI038, Postojna)]"
5,64613b29e32ccd3c085ab748,https://www.rtvslo.si/kultura/film-in-tv/zvono...,[Nadina Štefančič],2023-01-31T11:37:00,"""Zvonovi so zapisani v glasbo, literaturo, zvo...","""Oboje pa povezuje izročilo prednikov z našo d...","[{'caption': 'Zvonovi so glasbeni inštrumenti,...","""Naslov novega dokumentarnega filma opozarja n...","[Dominik Malovrh, Denis Božović, Bernard Perme...",kultura,"[Televizija Slovenija, zvonovi, Zven stoletij,...","[dokumentarni film, slovenski zvonovi, dedišči...",656251,7.0,kultura,povezuje izročilo prednikov našo dobo tradicij...,"[SI031, SI043, SI034, SI041, SI042]","[(SI043, Gorje), (SI043, Sveta Gora), (SI041, ..."
6,64613b29e32ccd3c085ab749,https://www.rtvslo.si/sport/nogomet/nemsko-nog...,[A. G.],2023-05-13T20:39:00,Bayern z Müllerjem v začetni postavi končno za...,Trener Bayerna Thomas Tuchel je ves teden posl...,[{'caption': 'Thomas Müller je na obeh tekmah ...,"Napet boj za ""solatni krožnik"" se nadaljuje. D...","[32. krog:, UNION BERLIN, BORUSSIA (D), Boruss...",sport,"[Thomas Müller, Serge Gnabry, Mathys Tel, Seba...","[Bayern, Dortmund, Bundesliga, Borussia, Thoma...",668069,8.0,NaN,trener bayerna thomas tuchel teden poslušal th...,[SI032],"[(SI032, Strelci)]"
14,64613b29e32ccd3c085ab751,https://www.rtvslo.si/slovenija/pirc-musar-slo...,[G. C.],2023-05-13T14:07:00,Pirc Musar: Slovenski vojaki si zaslužijo našo...,"""Zdi se, da smo prehitro pozabili občutke pono...",[{'img': 'https://img.rtvcdn.si/_up/upload/202...,"""Zdi se mi pomembno, da se znova začnejo tkati...","[Marjan Šarec, Robert Glavaš, Rok Ernecl, Matj...",slovenija,"[Koncert, Slovenska vojska, dan, predstavitev,...","[vojska, družba, povezave, razumevanje, naklon...",668040,68.0,NaN,zdi prehitro pozabili občutke ponosa varnosti ...,[SI041],"[(SI041, Ljubljana)]"
18,64613b29e32ccd3c085ab755,https://www.rtvslo.si/sport/kolesarstvo/dirka-...,[T. J.],2023-05-11T12:45:00,"Sprint v Neaplju dobil Pedersen, Roglič 15 km ...","""Zelo sem vesel. To je tisto, po kar smo prišl...",[{'caption': 'Mads Pedersen je prvič v karieri...,Mads Pedersen (Trek Segafredo) je zmagovalec 6...,"[Alexandre Delettre, Francesco Gavazzi, Simon ...",sport,"[Dirka po Italiji, 6. etapa, Neapelj, Roglič, ...","[kolesarstvo, zmaga, dirka, etapa, zmagovalka,...",667744,355.0,NaN,vesel prišli zmagati ekipo oddolžil zmago konc...,[SI034],"[(SI034, Klanc)]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73446,69cb7ec420cb54d19956df2e,https://www.rtvslo.si/stevilke/pogovor-lahko-r...,[Slavko Jerič],2026-03-31T06:34:11,Pogovor lahko reši življenje: Vanja Gomboc o s...,V 12. sezoni se ukvarjamo z željami. Izhodišče...,[{'caption': 'Igra asociacij in ... Vanja Gomb...,"""Še vedno sta napačni prepričanji, da bomo, če...","[Slavko Jerič, Vanja Gomboc, osebnega zdravnik...",stevilke,"[Vanja Gomboc, samomor, mediji, preprečevanje ...",NaN,776014,NaN,NaN,sezoni ukvarjamo željami izhodišče tokratne ep...,"[SI041, SI037]","[(SI037, Morava), (SI041, Rakitna)]"
73450,69cb7ec420cb54d19956df32,https://www.rtvslo.si/zabava-in-slog/avtomobil...,[Luka Gregorič (Avtomobilnost)],2026-03-31T08:05:47,Lexus RZ: Prvi pravi test krmiljenja prek žice,Je to le nepotreben tehnični zaplet ali največ...,[{'caption': 'V Avtomobilnosti smo imeli prvi ...,"Pozab

In [15]:
#novice_z_naselji_df = df[df['intersected_regions'].astype(bool)]



In [16]:
novice_z_naselji_df.count()

_id                    28579
url                    28579
authors                27921
date                   28579
title                  28579
paragraphs             28579
figures                28579
lead                   28560
mention                28579
topics                 28573
keywords               28579
gpt_keywords           11236
id                     28560
n_comments             18786
category                  96
clean_text             28579
intersected_regions    28579
intersected_naselja    28579
dtype: int64

In [17]:
novice_z_naselji_df_za_db = novice_z_naselji_df[["id", "title", "url", "date", "topics", "paragraphs", "intersected_regions", "clean_text", "intersected_naselja"]]

## Vectorize with TFIDF paragraphs

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vec = TfidfVectorizer(max_features=500, max_df=0.3, min_df=3) # Vektorji velikosti 500, max_df=0.3 => če se beseda pojavi 30%+ časa jo ignoriraj, min_df = minimalno kolikokrat se rabi beseda pojavit
tfidf_matrix = tfidf_vec.fit_transform(novice_z_naselji_df_za_db['clean_text'].fillna(''))
novice_z_naselji_df_za_db['tfidf'] = list(tfidf_matrix.toarray())

In [19]:
feature_names = tfidf_vec.get_feature_names_out()
np.save('final_data/tfidf_vocab.npy', feature_names, allow_pickle=True)

In [20]:
novice_z_naselji_df_za_db.columns

Index(['id', 'title', 'url', 'date', 'topics', 'paragraphs',
       'intersected_regions', 'clean_text', 'intersected_naselja', 'tfidf'],
      dtype='str')

# Save to db

In [21]:
import sqlite3
print(sqlite3.sqlite_version)

3.45.3


In [22]:
connection = sqlite3.connect('final_data/novice.db') # Naredi db če še ne obstaja
connection.execute("PRAGMA foreign_keys = ON")
cursor = connection.cursor()

In [23]:
# Naredi db
#cursor.execute("DELETE FROM novice")
#cursor.execute("DELETE FROM regije")

cursor.execute('''
    CREATE TABLE IF NOT EXISTS regije (
        id CHAR(5) PRIMARY KEY,
        name VARCHAR(50)
    )
''')
cursor.executemany("INSERT OR IGNORE INTO regije (id, name) VALUES (?, ?)", df_regije)

cursor.execute('''
    CREATE TABLE IF NOT EXISTS novice (
        id INTEGER PRIMARY KEY,
        title TEXT,
        url TEXT,
        date DATE,
        topic VARCHAR(30),
        content TEXT,
        clean_content TEXT,
        tfidf BLOB
    )
''')

# 2. Junction table for Regions
cursor.execute('''
    CREATE TABLE IF NOT EXISTS novice_regije (
        novica_id INTEGER,
        regija_id CHAR(5),
        PRIMARY KEY (novica_id, regija_id),
        FOREIGN KEY (novica_id) REFERENCES novice (id) ON DELETE CASCADE,
        FOREIGN KEY (regija_id) REFERENCES regije (id) ON DELETE CASCADE
    )
''')

# 3. Junction table for Naselja (includes regija_id to preserve the tuple)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS novice_naselja (
        novica_id INTEGER,
        regija_id CHAR(5),
        naselje VARCHAR(100),
        PRIMARY KEY (novica_id, regija_id, naselje),
        FOREIGN KEY (novica_id) REFERENCES novice (id) ON DELETE CASCADE,
        FOREIGN KEY (regija_id) REFERENCES regije (id) ON DELETE CASCADE
    )
''')

cursor.execute("CREATE INDEX IF NOT EXISTS idx_novice_topic ON novice (topic)")
cursor.execute("CREATE INDEX IF NOT EXISTS idx_novice_date ON novice (date)")
connection.commit()

In [24]:
import pickle
import sqlite3
import json

def save_data_to_db(connection, df):
    cursor = connection.cursor()
    
    novice_items = []
    regije_junction_items = []
    naselja_junction_items = []
    
    for _, row in df.iterrows():
        n_id = row['id']
        
        # 1. Convert TFIDF numpy array into a SQLite-compatible BLOB binary
        tfidf_blob = sqlite3.Binary(pickle.dumps(row['tfidf'])) if row['tfidf'] is not None else None
        
        # 2. Safely stringify the topics column (handling lists/strings)
        topics_val = row['topics']
        if isinstance(topics_val, list):
            # Option A: Convert to comma-separated string e.g., "Šport, Kronika"
            # topics_str = ", ".join(str(t) for t in topics_val)
            # Option B: Store as valid JSON string so you can read it back as a list later
            topics_str = json.dumps(topics_val, ensure_ascii=False)
        else:
            topics_str = str(topics_val) if pd.notna(topics_val) else ""

        # Append data aligned precisely with the 'novice' table schema
        novice_items.append((
            n_id,
            row['title'],
            row['url'],
            row['date'],
            topics_str,
            row['paragraphs'],
            row['clean_text'],
            tfidf_blob
        ))
        
        # 3. Explode the intersected_regions list
        r_ids = row['intersected_regions']
        if isinstance(r_ids, list):
            for r_id in r_ids:
                regije_junction_items.append((n_id, r_id))
                
        # 4. Explode the intersected_naselja tuples list [(region_id, naselje), ...]
        nas_tuples = row['intersected_naselja']
        if isinstance(nas_tuples, list):
            for r_id, naselje in nas_tuples:
                naselja_junction_items.append((n_id, r_id, naselje))

    try:
        # Insert into main news table
        cursor.executemany('''
            INSERT OR REPLACE INTO novice (id, title, url, date, topic, content, clean_content, tfidf) 
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', novice_items)
        
        # Insert into novice_regije junction table
        cursor.executemany('''
            INSERT OR IGNORE INTO novice_regije (novica_id, regija_id) 
            VALUES (?, ?)
        ''', regije_junction_items)
        
        # Insert into novice_naselja junction table
        cursor.executemany('''
            INSERT OR IGNORE INTO novice_naselja (novica_id, regija_id, naselje) 
            VALUES (?, ?, ?)
        ''', naselja_junction_items)
        
        connection.commit()
        print(f"Successfully saved {len(df)} articles.")
        
    except Exception as e:
        connection.rollback()
        print(f"Error during save: {e}")
    finally:
        cursor.close()

#! UNCOMMENT IF NEED
save_data_to_db(connection, novice_z_naselji_df_za_db)

Successfully saved 28579 articles.


In [25]:
connection.close()

In [26]:
novice_z_naselji_df_za_db.count()

id                     28560
title                  28579
url                    28579
date                   28579
topics                 28573
paragraphs             28579
intersected_regions    28579
clean_text             28579
intersected_naselja    28579
tfidf                  28579
dtype: int64